# Streaming

<img src="./assets/LC_streaming.png" width="400">

流式传输可以减少生成数据与用户接收数据之间的延迟。与代理常用的有两种类型：

In [8]:
from langchain.agents import create_agent

In [9]:
agent = create_agent(
    model="openai:gpt-5",
    system_prompt="你是一个全栈的戏剧演员",
)

## No Streaming (invoke)

In [10]:
result = agent.invoke({"messages": [{"role": "user", "content": "给我讲一个笑话"}]})
print(result["messages"][1].content)

我朋友开了家时光穿越公司：昨天开业，明天倒闭，上周已经清算了。客服还承诺48小时前回复。


## values
You have seen this streaming mode in our examples so far. 

In [11]:
# Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "给我讲一个程序员的笑话"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

给我讲一个程序员的笑话
================================== Ai Message ==================================

为什么程序员总把万圣节和圣诞节搞混？因为 Oct 31 == Dec 25。（八进制31等于十进制25）


## messages
消息以逐个令牌的方式流式传输数据——延迟最低。这非常适合像聊天机器人这样的交互式应用。

In [12]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "给我写一首适合冬天的诗"}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")

冬日慢歌

今夜的雪，把喧哗放轻，
屋檐垂着一串未说完的光。
呼出的白雾，在唇边排练台词，
句子一短，就落在围巾上温暖成霜。

炉里的水学会了唱歌，
每一次沸腾，都是回家的信号。
窗外的月亮轻轻推门，
冷与静，像两位耐心的观众。

我沿着风的侧影走，
听见树根在地底翻书。
万物把心跳调到最慢，
把希望悄悄折进一粒种子。

如果明天更冷，也不要怕，
让我们把手掌合在一起，
捧着彼此的呼吸，
照亮一小步向春天的路。

## Tools can stream too!
流式传输通常是指在最终结果准备就绪之前将信息传递给用户。这在很多情况下都非常有用。`get_stream_writer` 写入器允许您轻松地从您创建的数据源流式传输自定义数据。

In [13]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """获取指定城市的天气信息."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"正在查找城市城市数据: {city}")
    writer(f"已获取城市数据: {city}")
    return f"{city} 总是阳光明媚！"


agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "深圳的天气怎么样？"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='深圳的天气怎么样？', additional_kwargs={}, response_metadata={}, id='e5e55182-20e4-4769-97d5-fea2a15e6745')]})
('values', {'messages': [HumanMessage(content='深圳的天气怎么样？', additional_kwargs={}, response_metadata={}, id='e5e55182-20e4-4769-97d5-fea2a15e6745'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 130, 'total_tokens': 154, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CcX5FSsgiaZZKU2ebUxLbJLteH3x4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--2c65512f-e207-4455-be80-82a88cdcfdb1-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '深圳'}, 'id': 'call_tVEKXyADYcCZSKmdO1

In [14]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "深圳的天气怎么样？"}]},
    stream_mode=["custom"],
):
    print(chunk)

('custom', '正在查找城市城市数据: 深圳')
('custom', '已获取城市数据: 深圳')


## 自己尝试不同的模式！
修改流模式和选择条件以产生不同的结果。

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "深圳的天气怎么样？"}]},
    stream_mode=["values", "custom"],
):
    if chunk[0] == "custom":
        print(chunk[1])